# 02 — Ablações

Cada entrada de `ABLATIONS` altera **um** argumento da baseline; os hiperparâmetros de treino são os mesmos (`DEFAULTS`) para todas. Rode o notebook 01 antes, porque a baseline entra na comparação.

In [ ]:
# Setup: no Colab (site ou extensão do VS Code) clona o repositório do GitHub
# e instala as dependências; localmente só ajusta o sys.path.
# USE_DRIVE guarda pesos, JSONs e figuras no Google Drive para sobreviverem ao
# fim da sessão (o notebook 04 precisa dos pesos).
import os, shutil, sys
USE_DRIVE = True
# Detecta o Colab pela pasta /content em vez de "google.colab": no kernel da
# extensão do VS Code esse módulo pode não estar carregado.
IN_COLAB = os.path.isdir("/content") or "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/deep-vision-classification"
    # O código que roda é o do GitHub, não o da sua máquina: faça push antes.
    if not os.path.exists(REPO):
        !git clone -q https://github.com/davimoljo/deep-vision-classification.git {REPO}
    else:
        !git -C {REPO} pull -q
    %cd {REPO}
    !pip install -q -r requirements.txt
    import torch
    # Sem GPU o treino cai para a CPU sem avisar: a baseline levou ~3 h em vez
    # de ~15 min. Melhor parar aqui do que descobrir no fim do treino.
    if not torch.cuda.is_available():
        raise RuntimeError("Servidor sem GPU. Remova este servidor e crie outro "
                           "do Colab com GPU (T4), depois rode esta célula de novo.")
    print("GPU:", torch.cuda.get_device_name(0))
    if USE_DRIVE:
        if not os.path.isdir("/content/drive/MyDrive"):
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:
                print("Não montou o Drive:", e)
                print("No VS Code use 'Colab: Mount Google Drive to Server...' e rode esta célula de novo.")
        if os.path.isdir("/content/drive/MyDrive"):
            os.environ["OUTPUT_ROOT"] = "/content/drive/MyDrive/deep-vision-classification"
            print("Saídas em", os.environ["OUTPUT_ROOT"])
            # Baixar o CIFAR-10 do site original leva ~15 min a cada servidor
            # novo; com uma cópia do arquivo no Drive, as próximas vezes levam
            # segundos. O torchvision reaproveita o .tar.gz se o MD5 bater.
            cache = os.path.join(os.environ["OUTPUT_ROOT"], "data", "cifar-10-python.tar.gz")
            local = os.path.join(REPO, "data", "cifar-10-python.tar.gz")
            os.makedirs(os.path.dirname(local), exist_ok=True)
            if os.path.exists(cache) and not os.path.exists(local):
                shutil.copy(cache, local)
            from torchvision import datasets
            datasets.CIFAR10(os.path.join(REPO, "data"), train=True, download=True)
            if not os.path.exists(cache) and os.path.exists(local):
                os.makedirs(os.path.dirname(cache), exist_ok=True)
                shutil.copy(local, cache)
        else:
            print("ATENÇÃO: sem Drive, resultados e pesos somem quando o servidor desligar.")
    ROOT = REPO
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT)

In [ ]:
from src.pipeline import (DEFAULTS, STRATEGIES, build_cnn, build_from_config,
                          build_pretrained, count_parameters, evaluate_on_test, fit,
                          get_loaders, list_results, load_best_weights, load_result,
                          result_exists)
from src.plots import comparison_table, per_class_accuracy, plot_confusion, plot_curves
DEFAULTS

In [ ]:
# PROPOSTA — confirmar com o grupo. O enunciado exige no mínimo 3 ablações;
# cada uma custa um treino completo da baseline, então cortem as que não forem
# entrar na discussão do relatório.
ABLATIONS = {
    "abl_blocks_2":    {"blocks": 2},
    "abl_blocks_4":    {"blocks": 4},
    "abl_filters_16":  {"base_filters": 16},
    "abl_filters_64":  {"base_filters": 64},
    "abl_kernel_5":    {"kernel_size": 5},
    "abl_convs_1":     {"n_convs": 1},
    "abl_convs_3":     {"n_convs": 3},
    "abl_no_bn":       {"use_bn": False},
    "abl_dropout_0":   {"dropout": 0.0},
    "abl_dropout_0.5": {"dropout": 0.5},
    "abl_fc_128":      {"fc_dim": 128},
    "abl_fc_512":      {"fc_dim": 512},
    "abl_gap":         {"global_pool": True},
    "abl_residual":    {"residual": True},
}
# Garante a regra "uma característica por vez" antes de gastar GPU.
assert all(len(change) == 1 for change in ABLATIONS.values())

## Treino

Runs já salvas são puladas, então dá para rodar de novo depois que a sessão do Colab cair.

In [ ]:
loaders = get_loaders(DEFAULTS["batch_size"], img_size=32, augment=True)
for run, change in ABLATIONS.items():
    if result_exists(run):
        print(f"{run}: já existe, pulando")
        continue
    fit(build_cnn(**change), loaders, run)

## Comparação (validação)

In [ ]:
runs = ["baseline"] + list(ABLATIONS)
comparison_table(runs, name="ablacoes").sort_values("Val acc (%)", ascending=False)

In [ ]:
plot_curves(["baseline", "abl_no_bn", "abl_dropout_0", "abl_residual"], name="curvas_ablacoes_regularizacao");